# Public-hull maneuvering derivatives and added mass

This notebook applies the unrestricted, infinite-depth double-body boundary-element method to KCS, KVLCC2, DTMB 5415, DTC, and the analytic Wigley hull. It compares independent sway/yaw potentials with Wang's strict yaw approximation, applies the documented Schmitz stern cutoff only to velocity-dependent integrals, and keeps acceleration integrals on the complete wetted hull. No free-surface elevation is plotted.

In [ ]:
using Revise
using Pkg

function marinehydro_root(start=pwd())
    directory = abspath(start)
    while true
        project = joinpath(directory, "Project.toml")
        if isfile(project) && occursin("name = \"MarineHydro\"", read(project, String))
            return directory
        end
        parent = dirname(directory)
        parent == directory && error("Run Jupyter from the MarineHydro.jl repository.")
        directory = parent
    end
end

project_root = marinehydro_root()
Pkg.activate(project_root)
using MarineHydro
using CairoMakie
CairoMakie.activate!()
include(joinpath(project_root, "validation", "public_hulls", "public_hull_validation.jl"))

## Solve the benchmark set

Run `validation/public_hulls/fetch_geometry.sh` once before executing this cell. The standard study keeps every hull below 500 panels and is suitable for interactive work. Set `fine = true` for the higher-resolution geometry and convergence check.

In [ ]:
fine = false
cases = public_hull_cases(; fine)
results = [analyze_public_hull(case) for case in cases]
[(row.hull, row.panels, row.volume_relative_error,
  row.sway_boundary_residual, row.yaw_boundary_residual) for row in results]

## Waterline-clipped hull meshes

All axes are normalized by $L_{pp}$. The positive-$x$ end is the bow. DTMB 5415 is clipped from the official formatted Plot3D surface, DTC from the public OpenFOAM STL, KCS and KVLCC2 from the NMRI workshop surfaces, and Wigley from its analytic equation.

In [ ]:
function panel_wireframe(mesh, length_scale)
    x = Float64[]; y = Float64[]; z = Float64[]
    for panel in 1:mesh.nfaces
        indices = mesh.faces[panel, :] .+ 1
        for index in (indices[1], indices[2], indices[3], indices[4], indices[1])
            push!(x, mesh.vertices[index, 1] / length_scale)
            push!(y, mesh.vertices[index, 2] / length_scale)
            push!(z, mesh.vertices[index, 3] / length_scale)
        end
        push!(x, NaN); push!(y, NaN); push!(z, NaN)
    end
    return x, y, z
end

geometry_figure = Figure(size=(1600, 900), backgroundcolor=:white)
for (index, case) in enumerate(cases)
    row = fld(index - 1, 3) + 1
    column = mod(index - 1, 3) + 1
    length_scale = case.reference["length_m"]
    wire = panel_wireframe(case.mesh, length_scale)
    axis = Axis3(geometry_figure[row, column]; title="$(case.name), $(case.mesh.nfaces) panels",
        xlabel="x/Lpp", ylabel="y/Lpp", zlabel="z/Lpp",
        aspect=:data, azimuth=1.18pi, elevation=0.17pi)
    lines!(axis, wire...; color=(:steelblue4, 0.72), linewidth=0.55)
    scatter!(axis, case.mesh.centers[:, 1] ./ length_scale,
        case.mesh.centers[:, 2] ./ length_scale,
        case.mesh.centers[:, 3] ./ length_scale;
        color=case.mesh.centers[:, 1] ./ length_scale, colormap=:balance, markersize=2.5)
end
Label(geometry_figure[0, :], "Public hull meshes: bow is positive x"; fontsize=24, font=:bold)
output_directory = joinpath(project_root, "validation", "public_hulls", "results")
mkpath(output_directory)
save(joinpath(output_directory, "public_hull_geometry.png"), geometry_figure; px_per_unit=1.5)
geometry_figure

## Static-drift derivative comparison

The bars use standard MMG force and moment scales. Reference markers are captive-test values where available and Kijima's empirical estimate for Wigley. The strict and independent methods share the sway potential, so $Y_\beta'$ is identical; the independent yaw solution modifies $N_\beta'$.

In [ ]:
hull_names = [row.hull for row in results]
positions = collect(1:length(results))
derivative_figure = Figure(size=(1500, 620), backgroundcolor=:white)
Y_axis = Axis(derivative_figure[1, 1]; title="Sway drift derivative",
    ylabel="Yβ′", xticks=(positions, hull_names), xticklabelrotation=pi / 8)
N_axis = Axis(derivative_figure[1, 2]; title="Yaw drift derivative",
    ylabel="Nβ′", xticks=(positions, hull_names), xticklabelrotation=pi / 8)
barplot!(Y_axis, positions, [row.Y_beta_mmg for row in results];
    color=:steelblue3, width=0.58, label="independent BEM")
barplot!(N_axis, positions .- 0.18, [row.N_beta_mmg for row in results];
    color=:steelblue3, width=0.34, label="independent yaw")
barplot!(N_axis, positions .+ 0.18, [row.strict_N_beta_mmg for row in results];
    color=:darkorange2, width=0.34, label="strict Wang yaw")
finite_Y = findall(row -> isfinite(row.reference_Y_beta_mmg), results)
finite_N = findall(row -> isfinite(row.reference_N_beta_mmg), results)
scatter!(Y_axis, positions[finite_Y], [results[i].reference_Y_beta_mmg for i in finite_Y];
    color=:black, marker=:diamond, markersize=15, label="reference")
scatter!(N_axis, positions[finite_N], [results[i].reference_N_beta_mmg for i in finite_N];
    color=:black, marker=:diamond, markersize=15, label="reference")
axislegend(Y_axis; position=:lt)
axislegend(N_axis; position=:lt)
save(joinpath(output_directory, "public_hull_derivatives.png"), derivative_figure; px_per_unit=1.5)
derivative_figure

## Added-mass comparison

The first two panels use the MMG definitions $m_y'=A_{22}/(\tfrac12\rho L^2T)$ and $J_z'=A_{66}/(\tfrac12\rho L^4T)$. The third uses $A_{22}/(\rho\nabla)$, which permits the approximate DTC deep-water reference to be shown. Acceleration reciprocity and strict-versus-independent yaw differences remain in the CSV output.

In [ ]:
mass_figure = Figure(size=(1900, 620), backgroundcolor=:white)
my_axis = Axis(mass_figure[1, 1]; title="Sway added mass", ylabel="mᵧ′",
    xticks=(positions, hull_names), xticklabelrotation=pi / 8)
Jz_axis = Axis(mass_figure[1, 2]; title="Yaw added inertia", ylabel="Jz′",
    xticks=(positions, hull_names), xticklabelrotation=pi / 8)
volume_axis = Axis(mass_figure[1, 3]; title="Displacement normalization",
    ylabel="A₂₂/(ρ∇)", xticks=(positions, hull_names), xticklabelrotation=pi / 8)
barplot!(my_axis, positions, [row.m_y_prime for row in results]; color=:seagreen3, width=0.58)
barplot!(Jz_axis, positions .- 0.18, [row.J_z_prime for row in results];
    color=:seagreen3, width=0.34, label="independent yaw")
barplot!(Jz_axis, positions .+ 0.18, [row.strict_J_z_prime for row in results];
    color=:darkorange2, width=0.34, label="strict Wang yaw")
barplot!(volume_axis, positions, [row.A22_over_rho_displacement for row in results];
    color=:mediumpurple3, width=0.58)
finite_my = findall(row -> isfinite(row.reference_m_y_prime), results)
finite_Jz = findall(row -> isfinite(row.reference_J_z_prime), results)
finite_A22 = findall(row -> isfinite(row.reference_A22_over_rho_displacement), results)
scatter!(my_axis, positions[finite_my], [results[i].reference_m_y_prime for i in finite_my];
    color=:black, marker=:diamond, markersize=15)
scatter!(Jz_axis, positions[finite_Jz], [results[i].reference_J_z_prime for i in finite_Jz];
    color=:black, marker=:diamond, markersize=15, label="reference")
scatter!(volume_axis, positions[finite_A22],
    [results[i].reference_A22_over_rho_displacement for i in finite_A22];
    color=:black, marker=:diamond, markersize=15)
axislegend(Jz_axis; position=:rt)
save(joinpath(output_directory, "public_hull_added_mass.png"), mass_figure; px_per_unit=1.5)
mass_figure

## Interpretation

Added mass is the strongest validation target because it follows directly from the zero-frequency potential. The Schmitz-truncated velocity derivatives mix ideal-flow pressure with a geometry-dependent stern cutoff and should be treated as a low-order maneuvering estimate. Captive-test slopes also contain viscous cross-flow and finite-Froude-number effects absent from this double-body model. The CSV records those distinctions, both boundary residuals, geometry errors, and acceleration reciprocity.